# Asistente (Whisper + LLM)
Procesa los trabajos `asist_*.zip` que deja la UI en Drive (`PoC-Labelling/asistente`).
1. Entorno de ejecucion: **T4 GPU**.
2. Celda 1 una vez por sesion. Celda 2 cada vez que haya trabajos. Celda 3: diagnostico.

Fuente de verdad: `notebooks/asistente.ipynb` del repo (sincronizar con `python xd.py cuadernos`).

In [ ]:
import os, shutil, subprocess, time, urllib.request
from google.colab import drive
if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')
BUZON = '/content/drive/MyDrive/PoC-Labelling/asistente'
MODELO = 'qwen2.5:14b'      # defecto (buzon.MODELO_LLM); un trabajo puede pedir otro
# versiones fijas: las que funcionaron en L.6 (30/10/2026)
subprocess.run("pip -q install faster-whisper==1.2.1 ctranslate2==4.8.4 nvidia-cublas-cu12 "
               "'nvidia-cudnn-cu12==9.*' nvidia-cuda-runtime-cu12", shell=True, check=True)
if not shutil.which('ollama'):
    subprocess.run("apt-get -qq install -y zstd lshw > /dev/null && curl -fsSL https://ollama.com/install.sh | sh",
                   shell=True, check=True)

def vivo():
    try:
        urllib.request.urlopen('http://127.0.0.1:11434/api/version', timeout=2)
        return True
    except Exception:
        return False

if not vivo():
    subprocess.Popen(['ollama', 'serve'], stdout=open('/content/ollama.log', 'w'), stderr=subprocess.STDOUT)
    for _ in range(30):
        if vivo():
            break
        time.sleep(1)
print('ollama vivo:', vivo())
subprocess.run(['ollama', 'pull', MODELO], check=True)

In [ ]:
import json, shutil, sys, traceback, zipfile
from pathlib import Path

for zp in sorted(Path(BUZON).glob('asist_*.zip')):
    res, err = zp.with_name(zp.stem + '_resultado.json'), zp.with_name(zp.stem + '_error.txt')
    if res.exists() or err.exists():
        continue
    d = Path('/content/asist') / zp.stem
    shutil.rmtree(d, ignore_errors=True)
    d.mkdir(parents=True)
    zipfile.ZipFile(zp).extractall(d)
    for m in [p.stem for p in (d / 'codigo').glob('*.py')]:     # siempre el codigo que trae el zip
        sys.modules.pop(m, None)
    sys.path = [p for p in sys.path if not p.startswith('/content/asist/')]
    sys.path.insert(0, str(d / 'codigo'))
    print('=====', zp.name)
    try:
        import llm_ollama, trabajo
        llm_ollama.asegurar(json.loads((d / 'entrada.json').read_text(encoding='utf-8')).get('modelo_llm') or MODELO)
        r = trabajo.procesar(d, zp.name)
        tmp = res.with_suffix('.tmp')                              # nunca un resultado a medias
        tmp.write_text(json.dumps(r, ensure_ascii=True, indent=1), encoding='utf-8')
        tmp.replace(res)
        print('stt:', r['stt'])
        print(trabajo.asistente.resumen(r['resultado']))
        print('->', res.name)
    except Exception:
        traceback.print_exc()
        err.write_text(traceback.format_exc(), encoding='utf-8')
print('listo')

In [ ]:
!nvidia-smi --query-gpu=name,memory.used,memory.total --format=csv
!ollama ps
!grep -h "api/chat" /content/ollama.log /tmp/ollama_serve.log 2>/dev/null | tail -n 5